# Sentiment Analysis & Trading Signal Pipeline

**Analyst: Hector**  
**Course: CAP 3764 Advanced Data Science - FIU**

This notebook merges sentiment data with price data, calculates correlations, designs a trading signal, and creates visualizations.

## Prerequisites
- **Required**: `data/sentiment_articles.csv` (created by Kevin's sentiment_analysis.py)
- **Required**: `data/prices.csv` (collected by Data Engineer)
- The notebook will automatically handle column name differences (sentiment_compound vs vader_compound)

## Overview
- Load sentiment and price data
- Merge datasets on ticker and date
- Calculate daily returns and next-day returns
- Analyze correlation between sentiment and returns
- Design a simple threshold-based trading signal
- Create visualizations to explore the relationship

## Section 1: Setup & Data Loading

First, we import the necessary libraries and load our data files. We'll load the processed sentiment data (from Kevin's analysis) and the raw price data.

In [ ]:
# Import required libraries
# Using only: pandas, numpy, matplotlib, seaborn, scipy as specified
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Set plotting style for clean, professional charts
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

In [ ]:
# Load the processed sentiment data (created by Kevin's sentiment_analysis.py)
# Note: If the file has 'sentiment_compound' instead of 'vader_compound', we'll handle it
sentiment_path = '../data/sentiment_articles.csv'
sentiment_df = pd.read_csv(sentiment_path)

# Check column names and rename if needed
# Kevin's script outputs 'sentiment_compound', but instructions specify 'vader_compound'
if 'sentiment_compound' in sentiment_df.columns and 'vader_compound' not in sentiment_df.columns:
    sentiment_df = sentiment_df.rename(columns={'sentiment_compound': 'vader_compound'})

print(f"Sentiment data shape: {sentiment_df.shape}")
print(f"\nSentiment columns: {list(sentiment_df.columns)}")
print("\nFirst 5 rows of sentiment data:")
sentiment_df.head()

In [ ]:
# Load the raw price data (collected by Data Engineer)
prices_path = '../data/prices.csv'
prices_df = pd.read_csv(prices_path)

print(f"Price data shape: {prices_df.shape}")
print(f"\nPrice columns: {list(prices_df.columns)}")
print("\nFirst 5 rows of price data:")
prices_df.head()

In [ ]:
# Parse dates to ensure proper merging
# Convert date columns to datetime format for both dataframes
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'], errors='coerce')
prices_df['date'] = pd.to_datetime(prices_df['date'], errors='coerce')

# Check for any date parsing issues
print(f"Sentiment date range: {sentiment_df['date'].min()} to {sentiment_df['date'].max()}")
print(f"Price date range: {prices_df['date'].min()} to {prices_df['date'].max()}")
print(f"\nSentiment data - Missing dates: {sentiment_df['date'].isna().sum()}")
print(f"Price data - Missing dates: {prices_df['date'].isna().sum()}")

## Section 2: Data Merging

Now we merge the sentiment data with price data. We'll use an inner join on both ticker and date to ensure we only keep records where we have both sentiment and price information for the same stock on the same day.

In [ ]:
# Before merging, we need to aggregate sentiment data by ticker and date
# Multiple articles per day should be averaged
# Group by ticker and date, then take the mean of vader_compound scores
sentiment_daily = sentiment_df.groupby(['ticker', 'date']).agg({
    'vader_compound': 'mean',  # Average sentiment for multiple articles per day
    'headline': 'first'  # Keep one headline for reference
}).reset_index()

print(f"Daily sentiment shape (after aggregation): {sentiment_daily.shape}")
print(f"Unique tickers: {sentiment_daily['ticker'].unique()}")
print(f"Date range: {sentiment_daily['date'].min()} to {sentiment_daily['date'].max()}")

In [ ]:
# Merge sentiment and price data using inner join
# This keeps only records where we have both sentiment and price data
merged_df = pd.merge(
    sentiment_daily[['ticker', 'date', 'vader_compound']],  # Select only needed columns
    prices_df,
    on=['ticker', 'date'],
    how='inner'  # Inner join: only keep matching records
)

print(f"Merged data shape: {merged_df.shape}")
print(f"\nMerged data columns: {list(merged_df.columns)}")
print("\nFirst 5 rows of merged data:")
merged_df.head()

In [ ]:
# Save the merged dataset to data folder
output_path = '../data/merged_data.csv'
merged_df.to_csv(output_path, index=False)
print(f"Saved merged data to {output_path}")
print(f"Total records: {len(merged_df)}")
print(f"Unique tickers: {merged_df['ticker'].nunique()}")
print(f"Date range: {merged_df['date'].min()} to {merged_df['date'].max()}")

## Section 3: Calculate Daily Returns

We need to calculate daily returns (percentage change in price) and next-day returns. The next-day return is what we're trying to predict - it represents the return we would get if we traded based on today's sentiment.

In [ ]:
# Sort by ticker and date to ensure proper calculation of returns
# Returns must be calculated in chronological order for each stock
merged_df = merged_df.sort_values(['ticker', 'date']).reset_index(drop=True)

print("Data sorted by ticker and date")
print(f"First few dates per ticker:")
print(merged_df.groupby('ticker')['date'].agg(['min', 'max', 'count']))

In [ ]:
# Calculate daily return: percentage change from previous day's close
# Group by ticker to ensure we don't calculate returns across different stocks
merged_df['daily_return'] = merged_df.groupby('ticker')['close'].pct_change()

print("Daily returns calculated")
print(f"Daily return statistics:")
print(merged_df['daily_return'].describe())

In [ ]:
# Calculate next-day return: the return we would get if we traded based on today's sentiment
# Shift daily_return by -1 to get tomorrow's return for today's row
# Group by ticker to ensure we don't shift across different stocks
merged_df['next_day_return'] = merged_df.groupby('ticker')['daily_return'].shift(-1)

print("Next-day returns calculated")
print(f"Next-day return statistics:")
print(merged_df['next_day_return'].describe())

In [ ]:
# Drop rows where next_day_return is NaN (last day of each stock has no next day)
# Also drop rows where daily_return is NaN (first day of each stock)
merged_df = merged_df.dropna(subset=['daily_return', 'next_day_return'])

print(f"After dropping NaN rows: {merged_df.shape}")
print(f"Remaining records per ticker:")
print(merged_df['ticker'].value_counts())

## Section 4: Correlation Analysis

We'll calculate the Pearson correlation between sentiment scores and next-day returns for each stock. This tells us how strongly sentiment predicts future returns. We'll also calculate p-values to assess statistical significance.

In [ ]:
# Calculate correlation for each stock between sentiment and next-day returns
# Use scipy.stats.pearsonr to get both correlation coefficient and p-value
correlation_results = []

for ticker in merged_df['ticker'].unique():
    ticker_data = merged_df[merged_df['ticker'] == ticker]
    
    # Calculate Pearson correlation and p-value
    corr_coef, p_value = pearsonr(
        ticker_data['vader_compound'],
        ticker_data['next_day_return']
    )
    
    correlation_results.append({
        'ticker': ticker,
        'correlation': corr_coef,
        'p_value': p_value,
        'n_observations': len(ticker_data)
    })

# Convert to DataFrame for easy viewing
corr_df = pd.DataFrame(correlation_results)

print("Correlation Analysis Results:")
print("=" * 60)
corr_df

In [ ]:
# Format and display the correlation results more clearly
print("\n" + "=" * 60)
print("CORRELATION SUMMARY TABLE")
print("=" * 60)
for _, row in corr_df.iterrows():
    significance = "***" if row['p_value'] < 0.001 else "**" if row['p_value'] < 0.01 else "*" if row['p_value'] < 0.05 else ""
    print(f"{row['ticker']:6s} | Correlation: {row['correlation']:7.4f} | "
          f"P-value: {row['p_value']:6.4f} {significance} | "
          f"N: {int(row['n_observations'])}")
print("=" * 60)
print("Significance: *** p<0.001, ** p<0.01, * p<0.05")

In [ ]:
# Save correlation results to CSV
corr_output_path = '../data/correlation_results.csv'
corr_df.to_csv(corr_output_path, index=False)
print(f"Correlation results saved to {corr_output_path}")

## Section 5: Trading Signal (Threshold-Based)

We'll design a simple trading signal based on sentiment thresholds:
- **BUY**: If previous day's average sentiment > 0.3 (strongly positive)
- **SELL**: If previous day's average sentiment < -0.3 (strongly negative)  
- **HOLD**: Otherwise (neutral sentiment)

This is a simple rule-based signal that we can evaluate.

In [ ]:
# Define a simple function to generate trading signals based on sentiment threshold
def generate_signal(sentiment_score):
    """
    Generate trading signal based on sentiment threshold.
    BUY if sentiment > 0.3, SELL if < -0.3, else HOLD.
    """
    if sentiment_score > 0.3:
        return 'BUY'
    elif sentiment_score < -0.3:
        return 'SELL'
    else:
        return 'HOLD'

# Apply the signal function to each row
# We use the current day's sentiment to predict next day's return
merged_df['signal'] = merged_df['vader_compound'].apply(generate_signal)

print("Trading signals generated")
print(f"\nSignal distribution:")
print(merged_df['signal'].value_counts())

In [ ]:
# Count signals per stock
signal_counts = merged_df.groupby(['ticker', 'signal']).size().unstack(fill_value=0)
print("\nSignal counts per stock:")
print("=" * 40)
print(signal_counts)
print("\nTotal signals per stock:")
print(merged_df.groupby('ticker')['signal'].count())

## Section 6: Visualizations

We'll create three visualizations to explore the relationship between sentiment and stock prices:
1. Dual-axis chart showing sentiment and price over time
2. Scatter plot of sentiment vs next-day returns
3. Price chart with trading signals overlaid

In [ ]:
# Chart 1: Dual-axis - Sentiment + Price over time
# Select the stock with the most data points
ticker_counts = merged_df['ticker'].value_counts()
selected_ticker = ticker_counts.index[0]  # Stock with most data
ticker_data = merged_df[merged_df['ticker'] == selected_ticker].sort_values('date')

# Create figure with dual y-axes
fig, ax1 = plt.subplots(figsize=(12, 6))

# Left y-axis: Close price (line chart)
color = 'tab:blue'
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Close Price ($)', color=color, fontsize=12)
ax1.plot(ticker_data['date'], ticker_data['close'], color=color, linewidth=2, label='Close Price')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

# Right y-axis: Sentiment score (scatter plot)
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('VADER Compound Sentiment', color=color, fontsize=12)
ax2.scatter(ticker_data['date'], ticker_data['vader_compound'], 
           color=color, alpha=0.6, s=50, label='Sentiment')
ax2.tick_params(axis='y', labelcolor=color)
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Title and formatting
plt.title(f'Sentiment and Price Over Time: {selected_ticker}', fontsize=14, fontweight='bold')
fig.tight_layout()

# Display figure (not saving to keep structure clean)
print(f"Chart 1 displayed: sentiment and price for {selected_ticker}")
plt.show()

In [ ]:
# Chart 2: Scatter - Sentiment vs Next-Day Returns
# Create scatter plot colored by ticker
fig, ax = plt.subplots(figsize=(12, 6))

# Plot each ticker with different color
tickers = merged_df['ticker'].unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(tickers)))

for ticker, color in zip(tickers, colors):
    ticker_data = merged_df[merged_df['ticker'] == ticker]
    ax.scatter(ticker_data['vader_compound'], ticker_data['next_day_return'],
              label=ticker, alpha=0.6, s=50, color=color)

# Add regression line for all data
z = np.polyfit(merged_df['vader_compound'], merged_df['next_day_return'], 1)
p = np.poly1d(z)
ax.plot(merged_df['vader_compound'], p(merged_df['vader_compound']), 
       "r--", alpha=0.8, linewidth=2, label='Regression Line')

# Calculate overall correlation for annotation
overall_corr, _ = pearsonr(merged_df['vader_compound'], merged_df['next_day_return'])

# Labels and title
ax.set_xlabel('VADER Compound Sentiment', fontsize=12)
ax.set_ylabel('Next-Day Return', fontsize=12)
ax.set_title(f'Sentiment vs Next-Day Returns (Overall Correlation: {overall_corr:.4f})', 
            fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

# Display figure (not saving to keep structure clean)
print("Chart 2 displayed: sentiment vs next-day returns")
plt.show()

In [ ]:
# Chart 3: Signal overlay on price
# Use the same selected ticker as Chart 1
fig, ax = plt.subplots(figsize=(12, 6))

# Plot price line
ax.plot(ticker_data['date'], ticker_data['close'], 
       color='black', linewidth=2, label='Close Price', zorder=1)

# Overlay BUY signals (green triangles pointing up)
buy_signals = ticker_data[ticker_data['signal'] == 'BUY']
if len(buy_signals) > 0:
    ax.scatter(buy_signals['date'], buy_signals['close'],
              color='green', marker='^', s=150, alpha=0.7, 
              label='BUY Signal', zorder=2, edgecolors='darkgreen', linewidths=1)

# Overlay SELL signals (red triangles pointing down)
sell_signals = ticker_data[ticker_data['signal'] == 'SELL']
if len(sell_signals) > 0:
    ax.scatter(sell_signals['date'], sell_signals['close'],
              color='red', marker='v', s=150, alpha=0.7,
              label='SELL Signal', zorder=2, edgecolors='darkred', linewidths=1)

# Labels and title
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Close Price ($)', fontsize=12)
ax.set_title('Trading Signals Based on Sentiment Threshold', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Display figure (not saving to keep structure clean)
print(f"Chart 3 displayed: trading signals for {selected_ticker}")
plt.show()

## Section 7: Summary

### Key Findings

Based on the analysis above, here are the main findings:

In [ ]:
# Display summary statistics for interpretation
print("=" * 60)
print("ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nTotal merged records: {len(merged_df)}")
print(f"Stocks analyzed: {', '.join(merged_df['ticker'].unique())}")
print(f"\nDate range: {merged_df['date'].min().date()} to {merged_df['date'].max().date()}")

print("\n" + "-" * 60)
print("CORRELATION FINDINGS:")
print("-" * 60)
for _, row in corr_df.iterrows():
    print(f"{row['ticker']}: r = {row['correlation']:.4f}, p = {row['p_value']:.4f}")

print("\n" + "-" * 60)
print("TRADING SIGNAL DISTRIBUTION:")
print("-" * 60)
print(merged_df['signal'].value_counts())

print("\n" + "-" * 60)
print("SENTIMENT STATISTICS:")
print("-" * 60)
print(f"Mean sentiment: {merged_df['vader_compound'].mean():.4f}")
print(f"Std sentiment: {merged_df['vader_compound'].std():.4f}")
print(f"Min sentiment: {merged_df['vader_compound'].min():.4f}")
print(f"Max sentiment: {merged_df['vader_compound'].max():.4f}")

print("\n" + "-" * 60)
print("RETURN STATISTICS:")
print("-" * 60)
print(f"Mean next-day return: {merged_df['next_day_return'].mean():.4f}")
print(f"Std next-day return: {merged_df['next_day_return'].std():.4f}")
print("=" * 60)

### Written Summary

The analysis reveals the relationship between news sentiment and stock price movements. The correlation coefficients show how strongly sentiment predicts next-day returns for each stock. If correlations are weak (close to 0), this suggests that sentiment alone may not be a reliable predictor of short-term price movements. Strong positive correlations would indicate that positive sentiment tends to precede positive returns, while negative correlations would suggest contrarian behavior.

The trading signal based on sentiment thresholds provides a simple rule-based approach. The distribution of BUY, SELL, and HOLD signals shows how often each signal type occurs. If most signals are HOLD, it suggests that extreme sentiment (above 0.3 or below -0.3) is relatively rare in the dataset.

**Important Note**: The effectiveness of this trading signal would need to be validated through backtesting, which is beyond the scope of this analysis. The correlations and visualizations provide initial insights, but real-world trading decisions should consider transaction costs, market conditions, and other factors.